# Towards the Assembly of the BINGO Radiotelescope
## Auxiliary Notebook

**Author**: Luciano Barosi

**E-Mail**: lbarosi@df.ufcg.edu.br

**Date**: 2026-08-14

## Initialization

In [1]:
"""Imports e setup do ambiente."""

from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import nest_asyncio
import numpy as np  # noqa: F401
import matplotlib.pyplot as plt
# ── Detecta raiz do projeto e adiciona src/ ao path ANTES dos imports do pacote ──
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = None
for p in candidates:
    if (p / "src" / "radioskysims").exists() and (p / "settings.toml").exists():
        project_root = p
        break

if project_root is None:
    msg = "Raiz do projeto não encontrada."
    raise RuntimeError(msg)

src_path = str((project_root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── Imports do projeto (após configurar sys.path) ──
from radioskysims.domain.beam.properties import BeamProperties
from radioskysims.domain.simulation.pointing import build_pointing_track
from radioskysims.infrastructure.beams.loaders.csv import CSVBeamLoader
from radioskysims.infrastructure.config.config_loader import (
    build_radiotelescope_from_settings,
)
from radioskysims.application.services.sim_orchestrator import SimOrchestrator
from radioskysims.domain.common import TimeRange

#-------Plotting utilities----------------
from plotting import jai
from plotting.colors import MEASUREMENT, MODEL
#-----------------------------------------
nest_asyncio.apply()

/home/lbarosi/RadioSkySim/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:628: UserWarning: <built-in function allocate_lock> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


## Uirapuru

## Simulatiom

In [2]:
# ── Orquestrador BINGO ──
orch = SimOrchestrator.from_settings("uirapuru")


# ── Executa simulação multi-fonte (resultado LAZY — NUNCA materializar) ──
# O dataset simulation_result é sempre muito grande e não pode ser computado
# inteiro de uma vez. O orquestrador retorna arrays dask; events e to_hdf5/
# to_zarr consomem o dataset por chunks ao longo do tempo (memória limitada
# ao tamanho do chunk).
comp_result = orch.run(
    constellations=["GPS", "GALILEO", "GLONASS", "BEIDOU"],
    include_local_sources=True,  # Sol
    # include_radio_sources=True,  # NVSS (se houver dados)
    use_processor=True,
)


ds_orch = comp_result.simulation_result.dataset

{'event': 'spacetrack_update_empty_result', 'requested_count': 2, 'norad_ids': [67162, 68791], 'detail': 'SpaceTrack retornou 0 TLEs. Pode significar que: (1) todos os NORAD IDs já foram consultados hoje (cache diário); (2) as credenciais são inválidas; (3) não há TLEs disponíveis para os NORAD IDs solicitados no período.'}
{'event': 'spacetrack_update_empty_result', 'requested_count': 2, 'norad_ids': [67162, 68791], 'detail': 'SpaceTrack retornou 0 TLEs. Pode significar que: (1) todos os NORAD IDs já foram consultados hoje (cache diário); (2) as credenciais são inválidas; (3) não há TLEs disponíveis para os NORAD IDs solicitados no período.'}


In [3]:
comp_result.summary

<bound method SimulationOrchestratorResult.summary of ════════════════════════════════════════════════════════════════════════
  RADIOSKYSIM — RESUMO DA SIMULAÇÃO
════════════════════════════════════════════════════════════════════════

┌─ OBSERVER ──────────────────────────────────────────────
  Nome                    : uirapuru
  Localização (lat/lon)    : -7.21 / -35.91  (height 530 m)
  Frequency Range (MHz)    : 1000 → 1250  (n_channels=5000)
  Central Frequency (MHz) : 1125
  Antenna                 : BINGO Horn
    Pointing (Alt/Az)      : 65.00° / 0.00°
    Polarizations         : ['H_COPOL']
    Beam FWHM (deg)       : min 6.75 / med — / max 14.25
    Directivity (dBi)     : min 22.8147 / med — / max 34.1659

┌─ SIMULATION ─────────────────────────────────────────────
  Timerange (UTC)         : 2025-02-27T00:00:00.000 → 2025-02-28T00:00:00.000  (step 120 s)
  Sky ─ Localsources:
    - Sun
        Flux@fc (Jy)          : —
        Antenna Temp (K)      : —
        flux datafi

In [4]:
outdir = Path(project_root) / "data/TOD/"
outdir.mkdir(exist_ok=True)

# Nome com timestamp evita conflito/trava ao reexecutar (arquivo já existente)
ts = datetime.now().strftime("%Y%m%dT%H%M%S")

# HDF5 (NetCDF) — save_result (composto) salva dataset + sidecar JSON e
# atualiza os flags saved/filepath no config do SimulationOrchestratorResult.
# Sem chunk_s, o tamanho do chunk é derivado de target_mb (auto).
p_nc = outdir / f"tod_result_uirapuru_simulation_20250227_{ts}.nc"
comp_result.save_result(str(p_nc))

'/home/lbarosi/RadioSkySim/data/TOD/tod_result_uirapuru_simulation_20250227_20260818T175001.nc.json'

In [5]:
# ── Events: dataset de trânsitos calculados (por chunks, dataset pequeno) ──
# events percorre o dataset lazy em chunks ao longo do tempo, selecionando
# angle_boresight: cada intervalo contíguo no feixe vira um trânsito (fonte,
# início/fim, duração, T_antenna). O dataset de transits é pequeno e o
# dataset completo NUNCA é materializado.
events = comp_result.events

# ── Repositório de simulações (persistência do resumo no banco) ──
repo = orch.simulation_repository
sim_id = orch.save_result(comp_result, registered=True)

# O repositório atualiza o config do resultado com o que foi persistido:
comp = comp_result.config["computation"]
print(f"💾 Registro id={sim_id}")
print(f"   saved={comp.get('saved')} | registered={comp.get('registered')}")
print(f"   filepath={comp.get('filepath')}")

# ── Encerra o scheduler distribuído (se use_processor=True) ──
# Após persistir, libera workers/memória/dashboard do dask "distributed".
orch.shutdown()

💾 Registro id=12
   saved=True | registered=True
   filepath=/home/lbarosi/RadioSkySim/data/TOD/tod_result_uirapuru_simulation_20250227_20260818T175001.nc


In [ ]:
# TODO 02 — Trânsitos no feixe, órbitas no céu e waterfall
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from IPython.display import display

from radioskysims.domain.simulation.pointing import build_pointing_track
from radioskysims.infrastructure.config.config_loader import (
    build_radiotelescope_from_settings,
)

# ── 1) Tabela de intervalos de trânsito de satélites no feixe ──
transits = events.transits
if transits is not None and transits.sizes["transit"]:
    tr_df = (
        transits.to_dataframe()
        .reset_index()
        .sort_values("start_time")
        .reset_index(drop=True)
    )
    tr_df["start_utc"] = pd.to_datetime(tr_df["start_time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    tr_df["end_utc"] = pd.to_datetime(tr_df["end_time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    tr_df["duration_min"] = (tr_df["duration_s"] / 60.0).round(1)
    tr_df["t_antenna_mean"] = tr_df["t_antenna_mean"].round(3)
    tr_df["t_antenna_max"] = tr_df["t_antenna_max"].round(3)
    tr_df["angle_boresight_min_deg"] = tr_df["angle_boresight_min_deg"].round(2)

else:
    print("Nenhum trânsito detectado no feixe.")
    
display(
    tr_df[
        [
            "source", "source_type", "pointing",
            "start_utc", "end_utc", "duration_min",
            "t_antenna_mean", "t_antenna_max", "angle_boresight_min_deg",
        ]
    ]
)
print(
    f"Total: {len(tr_df)} trânsitos | "
    f"fontes no feixe: {', '.join(str(x) for x in events.objects_in_beam)}"
)

,source,source_type,pointing,start_utc,end_utc,duration_min,t_antenna_mean,t_antenna_max,angle_boresight_min_deg
0,GSAT0202 (GALILEO 6),satellite,0,2025-02-27 00:00:00,2025-02-27 00:08:00,8.0,0.003,0.250,0.34
1,GSAT0214 (GALILEO 18),satellite,0,2025-02-27 00:00:00,2025-02-27 00:04:00,4.0,27.951,2161.713,0.06
2,GPS BIIR-11 (PRN 19),satellite,0,2025-02-27 00:00:00,2025-02-27 00:10:00,10.0,0.002,0.662,0.38
3,BEIDOU-3 M9 (C29),satellite,0,2025-02-27 00:00:00,2025-02-27 00:32:00,32.0,0.005,0.554,0.25
4,COSMOS 2460 (732),satellite,0,2025-02-27 00:02:00,2025-02-27 00:40:00,38.0,0.002,2.478,0.39
...,...,...,...,...,...,...,...,...,...
58,BEIDOU-3 M10 (C30),satellite,0,2025-02-27 23:00:00,2025-02-28 00:00:00,60.0,1.455,180.813,0.16
59,COSMOS 2485 (747),satellite,0,2025-02-27 23:18:00,2025-02-27 23:44:00,26.0,0.002,0.513,0.42
60,COSMOS 2545 (760),satellite,0,2025-02-27 23:56:00,2025-02-28 00:00:00,4.0,0.001,0.382,0.42
61,BEIDOU-3 M25 (C47),satellite,0,2025-02-27 23:58:00,2025-02-28 00:00:00,2.0,0.001,0.128,0.44


Total: 63 trânsitos | fontes no feixe: BEIDOU-2 M3 (C11), BEIDOU-2 M6 (C14), BEIDOU-3 M10 (C30), BEIDOU-3 M12 (C26), BEIDOU-3 M13 (C32), BEIDOU-3 M15 (C34), BEIDOU-3 M16 (C35), BEIDOU-3 M20 (C42), BEIDOU-3 M21 (C43), BEIDOU-3 M23 (C45), BEIDOU-3 M25 (C47), BEIDOU-3 M27 (C49), BEIDOU-3 M28 (C50), BEIDOU-3 M3 (C21), BEIDOU-3 M5 (C23), BEIDOU-3 M9 (C29), COSMOS 2432 (719), COSMOS 2436 (723), COSMOS 2456 (730), COSMOS 2460 (732), COSMOS 2485 (747), COSMOS 2500 (755), COSMOS 2522 (752), COSMOS 2527 (756), COSMOS 2529 (757), COSMOS 2544 (759), COSMOS 2545 (760), COSMOS 2547 (705K), COSMOS 2559 (707K), COSMOS 2596 (708K), GPS BIIF-1 (PRN 25), GPS BIIF-12 (PRN 32), GPS BIIF-3 (PRN 24), GPS BIIF-8 (PRN 03), GPS BIII-3 (PRN 23), GPS BIII-5 (PRN 11), GPS BIII-6 (PRN 28), GPS BIII-7 (PRN 01), GPS BIIR-11 (PRN 19), GPS BIIR-13 (PRN 02), GPS BIIRM-1 (PRN 17), GPS BIIRM-2 (PRN 31), GPS BIIRM-3 (PRN 12), GSAT0103 (GALILEO-FM3), GSAT0202 (GALILEO 6), GSAT0203 (GALILEO 7), GSAT0206 (GALILEO 10), GSAT020

: 